# Two-Stage Training Notebook
This notebook demonstrates a two-stage training process for predicting athlete medals in the Olympics.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
import zipfile

# Load the dataset directly from the zip file
with zipfile.ZipFile('data/raw/athlete_events.zip') as z:
    with z.open('athlete_events.csv') as f:
        df = pd.read_csv(f)

# Normalize columns and types
df['sex'] = df['sex'].astype('category')
df['team'] = df['team'].astype('category')
df['noc'] = df['noc'].astype('category')
df['season'] = df['season'].astype('category')
df['city'] = df['city'].astype('category')
df['sport'] = df['sport'].astype('category')
df['event'] = df['event'].astype('category')
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['height'] = pd.to_numeric(df['height'], errors='coerce')
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')

# Feature set for Stage A
features_a = ['sex', 'age', 'height', 'weight', 'team', 'noc', 'year', 'season', 'city', 'sport', 'event']
df = df[features_a + ['medal']]

# Stage A: Binary classifier for has_medal
X_a = df[features_a]
y_a = df['medal'].notnull().astype(int)
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_a, y_a, stratify=y_a, test_size=0.2, random_state=42)

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(), ['age', 'height', 'weight']),
    ('cat', OneHotEncoder(), ['sex', 'team', 'noc', 'season', 'city', 'sport', 'event'])
])

# Pipeline for the model
pipeline_a = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
pipeline_a.fit(X_train_a, y_train_a)

# Reporting performance
y_pred_a = pipeline_a.predict(X_test_a)
print('Stage A Classification Report:')
print(classification_report(y_test_a, y_pred_a))

# Stage B: Multi-class classifier for medal types
medalists = df[df['medal'].notnull()]
X_b = medalists[features_a]
y_b = medalists['medal']
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, stratify=y_b, test_size=0.2, random_state=42)

pipeline_b = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
pipeline_b.fit(X_train_b, y_train_b)

# Reporting performance
y_pred_b = pipeline_b.predict(X_test_b)
print('Stage B Classification Report:')
print(classification_report(y_test_b, y_pred_b))

# Save the processed dataset
df.to_parquet('data/processed/athlete_events_model_ready.parquet', index=False)

# Save the models
joblib.dump(pipeline_a, 'models/stage_a_has_medal.joblib')
joblib.dump(pipeline_b, 'models/stage_b_medal_type.joblib')

## Conclusion
In this notebook, we successfully trained models to predict medal winning and medal types for athletes. We saved the dataset and models for future use.